# Lab | RAG from Scratch
## A privacy assistant for the GDPR

The bank from this week's labs has a data protection team, and it gets the same questions every day, from customers and from colleagues: *Can I ask you to delete my data? How long do we have to report a data breach? Do we need consent for this?* The answers are all in one document, the **General Data Protection Regulation (GDPR)**, Europe's data protection law, but it is 88 pages long.

Your job is to build a first version of a **privacy assistant**: it answers questions from the GDPR, cites the passages it used so the team can check them, and says so when the GDPR does not answer the question. Then you **evaluate** it on 10 questions with known answers, to decide whether it is good enough to show the team.

**This lab practises** retrieval-augmented generation. You will use, from today's **RAG from Scratch** lesson: loading a PDF, chunking with overlap and metadata, embeddings with MiniLM, a Chroma collection, retrieval with metadata filters, a grounded prompt with citations and refusals, and the hit rate and citation checks.

**By the end, you can:**

1. Implement a RAG pipeline (chunk, embed, store, retrieve, generate) over a new document.
2. Evaluate it with a test set, and explain whether each failure comes from retrieval or from generation.

Each challenge ends with a **checkpoint** that tells you what you should see. Retrieval gives the same results every time; the LLM's answers change a little from run to run. **The whole lab costs about one US cent** with the model below; the last cell shows your total.

**The document** is `data/gdpr.pdf`, [Regulation (EU) 2016/679](https://eur-lex.europa.eu/eli/reg/2016/679/oj) as published in the Official Journal, from EUR-Lex (© European Union, https://eur-lex.europa.eu, reused under [Commission Decision 2011/833/EU](https://eur-lex.europa.eu/eli/dec/2011/833/oj)). Nothing your assistant says is legal advice.

## Setup

You need `pip install pypdf sentence-transformers chromadb openai python-dotenv`. As in yesterday's lab, create a file called `.env` in the `your-code` folder with one line, `OPENAI_API_KEY=sk-...`. The repository's `.gitignore` already lists `.env`: **never write the key in the notebook**, and check with `git status` before you commit that `.env` is not listed.

The cell below loads the libraries, the OpenAI client, a `cost` function and a `track` function that logs the tokens of every call, as yesterday. It also gives you `load_pdf`, the loader from the lesson, with the noise pattern changed for this document: the GDPR prints a footer such as `4.5.2016 L 119/41 Official Journal of the European Union EN` on every page.

In [1]:
pip install pypdf sentence-transformers chromadb openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import logging
import warnings
from collections import Counter

os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")
pd.set_option("display.max_colwidth", 90)

load_dotenv()
print("Key found:", "OPENAI_API_KEY" in os.environ)
client = OpenAI()
MODEL = "gpt-5.4-nano"

PRICES = {"gpt-5.4-nano": (0.20, 1.25)}      # US dollars per million tokens (input, output), October 2026
usage_log = []

def cost(input_tokens, output_tokens):
    price_in, price_out = PRICES[MODEL]
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000

def track(response, step):
    """Record the tokens of a response under the name of a lab step, and return the response."""
    usage_log.append({"step": step, "input": response.usage.prompt_tokens, "output": response.usage.completion_tokens})
    return response

NOISE = re.compile(r"L 119/\d+ Official Journal of the European Union")

def load_pdf(path):
    """Return one dictionary per page, {"page": number, "text": text}, without the page footers."""
    pages = []
    for number, page in enumerate(PdfReader(path).pages, start=1):
        lines = [line.strip() for line in page.extract_text().split("\n")]
        lines = [line for line in lines if line and not NOISE.search(line)]
        pages.append({"page": number, "text": "\n".join(lines)})
    return pages

Key found: True


**Checkpoint:** `Key found: True`, and no error.

## Challenge 1: load and chunk the GDPR

1. Load `../data/gdpr.pdf` with `load_pdf`, and print the number of pages and the total number of words.
2. Find the page where **Article 17** (*Right to erasure*) starts, and print the first 600 characters of that page. Look at the text: is it clean enough to use?
3. Write `chunk_document(pages, size=150, overlap=30)`. It returns a list of chunks, each a dictionary with an `id` (`"gdpr-0"`, `"gdpr-1"`, ...), the `text`, the `page` where the chunk starts and its `section`: `"Recitals"` before Article 1, then `"Article 1"`, `"Article 2"`, and so on (the section most of the chunk's words come from). In the GDPR, every article starts with a line such as `Article 17`.
4. Chunk the document and print the number of chunks and the share that are recitals.
5. Load the embedding model with `embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")`, count the tokens of every chunk with `embedder.tokenizer`, and print the median and the share of chunks over the model's 256-token limit.

In [3]:
# 1. Load
reader = PdfReader("../data/gdpr.pdf")
print(len(reader.pages), "pages")


# 2. The page of Article 17
for page_num, page in enumerate(reader.pages):
    text = page.extract_text()

    if "Article 17" in text and "Right to erasure" in text:
        print(f"Article 17 starts on page {page_num + 1}")
        print("\nFirst 600 characters:\n")
        print(text[:600])
        break

# 3. Chunking function
def chunk_document(pages, size=150, overlap=30):
    words = []
    pages_info = []
    sections = []

    current_section = "Recitals"

    # Extract words and track their page and section
    for page_num, page in enumerate(pages, start=1):
        text = page.extract_text() or ""

        for line in text.splitlines():
            # Detect article headings
            match = re.match(r"^\s*Article\s+(\d+)\s*$", line, re.IGNORECASE)

            if match:
                current_section = f"Article {match.group(1)}"

            line_words = line.split()

            words.extend(line_words)
            pages_info.extend([page_num] * len(line_words))
            sections.extend([current_section] * len(line_words))

    #Creating overlapping chunks
    chunks = []
    step = size - overlap

    for start in range(0, len(words), step):
        end = min(start + size, len(words))

        chunk_words = words[start:end]
        chunk_sections = sections[start:end]

        if not chunk_words:
            break

        # Determine which section contributes the most words
        section = max(
            set(chunk_sections),
            key=lambda s: chunk_sections.count(s)
        )

        chunks.append({
            "id": f"gdpr-{len(chunks)}",
            "text": " ".join(chunk_words),
            "page": pages_info[start],
            "section": section
        })

        if end == len(words):
            break

    return chunks

# 4. Chunks and recitals
chunks = chunk_document(reader.pages, size=150, overlap=30)

n_chunks = len(chunks)
n_recitals = sum(chunk["section"] == "Recitals" for chunk in chunks)

print(f"\nTotal chunks: {n_chunks}")
print(f"Share of recitals: {n_recitals / n_chunks:.2%}")

#loading embedding model

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# 5. Tokens per chunk
token_counts = [
    len(embedder.tokenizer.encode(chunk["text"], truncation=False))
    for chunk in chunks
]

88 pages
Article 17 starts on page 43

First 600 characters:

Article 15 
Right of access by the data subject 
1. The data subject shall have the right to obtain from the controller confirmation as to whether or not personal data 
concerning him or her are being processed, and, where that is the case, access to the personal data and the following 
information: 
(a)  the purposes of the processing; 
(b)  the categories of personal data concerned; 
(c)  the recipients or categories of recipient to whom the personal data have been or will be disclosed, in particular 
recipients in third countries or international organisations; 
(d)  where possible, the env

Total chunks: 465
Share of recitals: 44.30%


In [4]:
#token statistics
median_tokens = np.median(token_counts)
share_over_limit = sum(count > 256 for count in token_counts) / n_chunks

print(f"Median tokens per chunk: {median_tokens:.1f}")
print(f"Share of chunks over 256 tokens: {share_over_limit:.2%}")


# Inspect a sample chunk
print("\nExample chunk:")
print(chunks[0])

Median tokens per chunk: 176.0
Share of chunks over 256 tokens: 0.00%

Example chunk:
{'id': 'gdpr-0', 'text': 'I (Legislative acts) REGULATIONS REGULATION (EU) 2016/679 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 27 April 2016 on the protection of natural persons with regard to the processing of personal data and on the free movement of such data, and repealing Directive 95/46/EC (General Data Protection Regulation) (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 16 thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), Having regard to the opinion of the Committee of the Regions (2), Acting in accordance with the ordinary legislative procedure (3), Whereas: (1) The protection 

**Checkpoint:** 88 pages and about 55,000 words. Article 17 starts on page 43, and the text is clean: the heading, the title *Right to erasure ('right to be forgotten')* and the numbered paragraphs, with no footer. About 458 chunks, of which about 45% are recitals (the GDPR has 173 recitals for 99 articles). A median of about 174 tokens per chunk and none over 256: 150-word chunks fit MiniLM.

## Challenge 2: index the chunks

1. Embed the text of every chunk with `embedder`, normalised, and print the shape of the result.
2. Create a Chroma collection called `"gdpr"` (in memory, with `chromadb.Client()`) that compares vectors with cosine distance, and add the ids, texts, embeddings and metadata (`page` and `section`) of all chunks. Print `collection.count()`.

In [5]:
# 1. Embeddings
texts = [chunk["text"] for chunk in chunks]

embeddings = embedder.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embeddings shape:", embeddings.shape)


# 2. Collection
client = chromadb.Client()

collection = client.get_or_create_collection(
    name="gdpr",
    metadata={"hnsw:space": "cosine"}
)


# 3. Prepare chunk data
ids = [chunk["id"] for chunk in chunks]

metadatas = [
    {
        "page": chunk["page"],
        "section": chunk["section"]
    }
    for chunk in chunks
]

# 4. Use=ing batches to avoid exceeding ChromaDB's batch limit
batch_size = 500

for i in range(0, len(chunks), batch_size):
    collection.add(
        ids=ids[i:i + batch_size],
        documents=texts[i:i + batch_size],
        embeddings=embeddings[i:i + batch_size].tolist(),
        metadatas=metadatas[i:i + batch_size]
    )


# 5. Check how many chunks were indexed
print("Chunks in collection:", collection.count())

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Embeddings shape: (465, 384)
Chunks in collection: 465


**Checkpoint:** `(458, 384)` and `458 chunks in the collection` (the same number as your chunks, whatever it is).

## Challenge 3: retrieve

1. Write `retrieve(question, k=5, where=None)`: it embeds the question with the same model, queries the collection, and returns a DataFrame with the columns `id`, `section`, `page`, `distance` and `text`.
2. Define `RULES_ONLY`, a `where` filter that excludes the recitals.
3. Retrieve the top 5 chunks for *Can I ask a company to delete my data?*, first without a filter and then with `RULES_ONLY`.

In [6]:
# 1. Retrieval function
def retrieve(question, k=5, where=None):

    # Embed the question using the same model
    query_embedding = embedder.encode(
        question,
        normalize_embeddings=True
    ).tolist()

    # Query ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where=where,
        include=["documents", "metadatas", "distances"]
    )

    # Convert results to a DataFrame
    df = pd.DataFrame({
        "id": results["ids"][0],
        "section": [m["section"] for m in results["metadatas"][0]],
        "page": [m["page"] for m in results["metadatas"][0]],
        "distance": results["distances"][0],
        "text": results["documents"][0]
    })

    return df

# 2. Filter
RULES_ONLY = {"section": {"$ne": "Recitals"}}

question = "Can I ask a company to delete my data?"

# 3. Without and with the filter
#without filter
print("WITHOUT FILTER:")
results_all = retrieve(question, k=5)
display(results_all)

#with Filter
print("WITH RULES_ONLY FILTER:")
results_rules = retrieve(question, k=5, where=RULES_ONLY)
display(results_rules)

WITHOUT FILTER:


,id,section,page,distance,text
0,gdpr-262,Article 17,44,0.469475,"Article 21(1) and there are no overriding legitimate grounds for the processing, or th..."
1,gdpr-42,Recitals,7,0.501616,to the processing of personal data and how to exercise their rights in relation to suc...
2,gdpr-261,Article 17,43,0.507977,subject shall have the right to obtain from the controller the erasure of personal dat...
3,gdpr-79,Recitals,13,0.535760,"freedom of expression and information, for compliance with a legal obligation, for the..."
4,gdpr-249,Article 13,41,0.541825,access to and rectification or erasure of personal data or restriction of processing c...


WITH RULES_ONLY FILTER:


,id,section,page,distance,text
0,gdpr-262,Article 17,44,0.469475,"Article 21(1) and there are no overriding legitimate grounds for the processing, or th..."
1,gdpr-261,Article 17,43,0.507977,subject shall have the right to obtain from the controller the erasure of personal dat...
2,gdpr-249,Article 13,41,0.541825,access to and rectification or erasure of personal data or restriction of processing c...
3,gdpr-451,Article 88,84,0.544131,"of employees' personal data in the employment context, in particular for the purposes ..."
4,gdpr-260,Article 16,43,0.553601,"electronic means, and unless otherwise requested by the data subject, the information ..."


**Checkpoint:** Without the filter, the first chunk comes from Article 15 and the other four are recitals. With `RULES_ONLY`, all five are articles, and Article 17 (page 43) is third.

**Question:** which article answers the question, and in which position does it appear with and without the filter? Why do the recitals compete with the articles?

Article 17 seems to answer the question "Can I ask a company to delete my data?". Without the filter, the answer appears on position 1 and 3, while the rest are articles or recitals that are not relevant to the answer. With filter, Article 17 appears on the first 2 positions and the following 3 positions seem to have cosine similarity to Article 17. 

The recitals compete with the Articles because without filter it was not defined not to obtain the answer from these parts and might have some information regarding the answert to the question.

## Challenge 4: grounded answers

1. Write the system prompt of the privacy assistant. It must answer **only** from the excerpts, cite the excerpt ids in square brackets after each statement (for example `[gdpr-12]`), reply exactly `I don't know based on the GDPR.` and nothing else when the excerpts do not contain the answer, and answer in 2 to 4 sentences of plain English.
2. Write `answer(question, k=5, where=RULES_ONLY)`. It retrieves the chunks, builds the context (each chunk labelled with its id, section and page, as in the lesson), calls the model with the system prompt and a user message containing the excerpts and the question, tracks the call as `"answers"`, and returns a dictionary with the `answer`, the `cited` ids (find them in the answer with a regular expression), the `retrieved` ids and their `sections`.
3. Print the answers to these three questions, each with the sections it cited:
   - *How long does a company have to report a data breach to the authority?*
   - *Can I get a copy of the personal data a company holds about me?*
   - *What is the VAT rate for online services in Spain?*

In [10]:
# 1. System prompt
SYSTEM_PROMPT = """
You are a privacy assistant answering questions about the GDPR.

Follow these rules strictly:
1. Answer ONLY using the provided GDPR excerpts.
2. Do not use outside knowledge or make assumptions.
3. Cite the excerpt ID in square brackets after every factual statement, for example [gdpr-12].
4. Use only excerpt IDs that appear in the provided context.
5. If the excerpts do not contain enough information to answer the question, reply exactly: I don't know based on the GDPR.
6. Keep your answer to 2 to 4 sentences of plain English, except when giving the exact fallback response.
7. Do not invent legal provisions or citations.
"""

# 2. Answer function
client_openai = OpenAI()

def answer(question, k=5, where=RULES_ONLY):

    # 1. Retrieve relevant GDPR chunks
    retrieved_df = retrieve(question, k=k, where=where)

    # 2. Build context with chunk ID, section and page
    context = "\n\n".join(
        f"[{row['id']}] {row['section']}, page {row['page']}\n{row['text']}"
        for _, row in retrieved_df.iterrows()
    )

    # 3. Call the model
    response = client_openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"GDPR excerpts:\n\n{context}\n\nQuestion: {question}"
            }
        ],
        temperature=0,
        extra_headers={"X-Tag": "answers"}
    )

    # 4. Extract the answer
    answer_text = response.choices[0].message.content

    # 5. Find cited chunk IDs using regex
    cited_ids = list(dict.fromkeys(
        re.findall(r"\[(gdpr-\d+)\]", answer_text)
    ))

    # 6. Return answer and metadata
    return {
        "answer": answer_text,
        "cited": cited_ids,
        "retrieved": retrieved_df["id"].tolist(),
        "sections": retrieved_df["section"].tolist()
    }

# 3. Three questions
questions = [
    "How long does a company have to report a data breach to the authority?",
    "Can I get a copy of the personal data a company holds about me?",
    "What is the VAT rate for online services in Spain?"
]

for question in questions:

    result = answer(question)

    print("=" * 70)
    print("QUESTION:", question)
    print("\nANSWER:", result["answer"])

    # Match cited IDs to their sections
    cited_sections = [
        section
        for chunk_id, section in zip(result["retrieved"], result["sections"])
        if chunk_id in result["cited"]
    ]


QUESTION: How long does a company have to report a data breach to the authority?

ANSWER: A company must notify the supervisory authority of a personal data breach without undue delay and, where feasible, not later than 72 hours after becoming aware of it, unless the breach is unlikely to result in a risk to the rights and freedoms of natural persons [gdpr-298].
QUESTION: Can I get a copy of the personal data a company holds about me?

ANSWER: Yes, you have the right to obtain a copy of the personal data undergoing processing by the controller. The controller must provide this information, and if you request further copies, they may charge a reasonable fee based on administrative costs [gdpr-259].
QUESTION: What is the VAT rate for online services in Spain?

ANSWER: I don't know based on the GDPR.


**Checkpoint:** The breach answer says *without undue delay* and, where feasible, *not later than 72 hours* after the company becomes aware of the breach, citing a chunk of Article 33. The copy answer says yes, citing Article 15 (and mentions that further copies can cost a reasonable fee). The VAT question gets `I don't know based on the GDPR.` (sometimes followed by chunk ids, even though the rule says *nothing else*). Your wording will differ.

**Question:** open `data/gdpr.pdf` at the page of one chunk your assistant cited, and compare. Is the answer faithful to the text? Did it leave out anything important?

The answers appear faithful to the GDPR provisions. This demonstrates that the RAG system can produce grounded answers, although the completeness of its responses depends on which chunks are retrieved.

## Challenge 5: evaluate the assistant

The data protection team wrote 10 questions and, for each one, the article that answers it. Use them to decide whether the assistant is ready to show.

1. Write `hit_rate(k=5, where=None)`: the share of test questions for which at least one of the top k chunks comes from the expected article. Print a table with the hit rate for `k` = 3 and 5, without a filter and with `RULES_ONLY`, and the questions missed in each case.
2. Run `answer` on the 10 questions and build a DataFrame with one row per question and four `True`/`False` columns: `right article retrieved`, `cites` (at least one id), `cites only what it saw` (every cited id was retrieved) and `cites the right article`. Print the mean of each column and show the rows that failed any check.
3. Run `answer` on the two out-of-scope questions in `OUT_OF_SCOPE` and count how many were refused.

In [11]:
TEST_SET = pd.DataFrame([
    ("Can I ask a company to delete my data?", "Article 17"),
    ("How long does a company have to tell the authority about a data breach?", "Article 33"),
    ("What are the legal grounds for processing personal data?", "Article 6"),
    ("How much can a company be fined under the GDPR?", "Article 83"),
    ("Can I get a copy of the personal data a company holds about me?", "Article 15"),
    ("At what age can a child consent to online services on their own?", "Article 8"),
    ("When does a company need a data protection officer?", "Article 37"),
    ("Can I move my data from one service provider to another?", "Article 20"),
    ("What counts as valid consent?", "Article 7"),
    ("Can a decision about me be made only by an algorithm?", "Article 22"),
], columns=["question", "expected"])
OUT_OF_SCOPE = ["What is the VAT rate for online services in Spain?",
                "How many vacation days do employees get in France?"]

# 1. Hit rate
def hit_rate(k=5, where=None):
    hits = 0
    missed = []

    for _, row in TEST_SET.iterrows():
        results = retrieve(row["question"], k=k, where=where)

        if row["expected"] in results["section"].values:
            hits += 1
        else:
            missed.append(row["question"])

    return hits / len(TEST_SET), missed

# Compare retrieval settings
hit_results = []

for k in [3, 5]:
    for filter_name, where in [
        ("No filter", None),
        ("RULES_ONLY", RULES_ONLY)
    ]:
        rate, missed = hit_rate(k=k, where=where)

        hit_results.append({
            "k": k,
            "filter": filter_name,
            "hit_rate": rate,
            "missed_questions": missed
        })

hit_df = pd.DataFrame(hit_results)

display(hit_df)

# 2. Answer checks
evaluation_results = []

for _, row in TEST_SET.iterrows():

    question = row["question"]
    expected = row["expected"]

    # Generate an answer
    result = answer(question, k=5, where=RULES_ONLY)

    retrieved_ids = result["retrieved"]
    retrieved_sections = result["sections"]
    cited_ids = result["cited"]

    # Map retrieved IDs to their sections
    id_to_section = dict(zip(retrieved_ids, retrieved_sections))

    # Check 1: Was the right article retrieved?
    right_article_retrieved = expected in retrieved_sections

    # Check 2: Does the answer contain at least one citation?
    cites = len(cited_ids) > 0

    # Check 3: Are all citations from retrieved chunks?
    cites_only_what_it_saw = set(cited_ids).issubset(
        set(retrieved_ids)
    )

    # Check 4: Does it cite the expected article?
    cites_right_article = any(
        id_to_section.get(chunk_id) == expected
        for chunk_id in cited_ids
    )

    evaluation_results.append({
        "question": question,
        "expected": expected,
        "right article retrieved": right_article_retrieved,
        "cites": cites,
        "cites only what it saw": cites_only_what_it_saw,
        "cites the right article": cites_right_article
    })


# Create evaluation DataFrame
eval_df = pd.DataFrame(evaluation_results)

# Calculate mean of each Boolean column
check_columns = [
    "right article retrieved",
    "cites",
    "cites only what it saw",
    "cites the right article"
]

print("Mean scores:")
print(eval_df[check_columns].mean())

# Show questions that failed at least one check
print("\nQuestions failing at least one check:")
display(eval_df[~eval_df[check_columns].all(axis=1)])

# 3. Out-of-scope questions
refusal_results = []

for question in OUT_OF_SCOPE:

    result = answer(question, k=5, where=RULES_ONLY)

    refused = (
        result["answer"].strip()
        == "I don't know based on the GDPR."
    )

    refusal_results.append({
        "question": question,
        "answer": result["answer"],
        "refused": refused
    })


refusal_df = pd.DataFrame(refusal_results)

print("Out-of-scope evaluation:")
display(refusal_df)

print(
    f"\nCorrect refusals: "
    f"{refusal_df['refused'].sum()} / {len(OUT_OF_SCOPE)}"
)

,k,filter,hit_rate,missed_questions
0,3,No filter,0.7,"[What are the legal grounds for processing personal data?, Can I move my data from one..."
1,3,RULES_ONLY,0.7,"[What are the legal grounds for processing personal data?, Can I move my data from one..."
2,5,No filter,0.7,"[What are the legal grounds for processing personal data?, Can I move my data from one..."
3,5,RULES_ONLY,0.8,"[Can I move my data from one service provider to another?, Can a decision about me be ..."


Mean scores:
right article retrieved    0.8
cites                      0.8
cites only what it saw     1.0
cites the right article    0.7
dtype: float64

Questions failing at least one check:


,question,expected,right article retrieved,cites,cites only what it saw,cites the right article
2,What are the legal grounds for processing personal data?,Article 6,True,False,True,False
7,Can I move my data from one service provider to another?,Article 20,False,False,True,False
9,Can a decision about me be made only by an algorithm?,Article 22,False,True,True,False


Out-of-scope evaluation:


,question,answer,refused
0,What is the VAT rate for online services in Spain?,I don't know based on the GDPR.,True
1,How many vacation days do employees get in France?,I don't know based on the GDPR.,True



Correct refusals: 2 / 2


**Checkpoint:** Hit rates of 0.5 and 0.8 at k = 3, and 0.6 and 0.8 at k = 5 (without and with the filter). With the filter, the two misses are the legal grounds (Article 6) and data portability (Article 20). In the answer checks, about 0.8 of the answers cite the right article and none cites an id it did not see, and the failures are the same two questions, both with `right article retrieved` = `False`. Both out-of-scope questions are refused. The retrieval numbers should match exactly; the answers can differ a little.

**Question:** for each failure, say whether it comes from retrieval or from generation, and why. Then decide: would you show this assistant to the data protection team? What would you change first, and what would you tell them about how to use it?

The evaluation identified three failures out of ten questions.  The Article 6 failure originated from generation, while Article 20 failed form retrieval and potentially generation, as the correct article was missing and the answer contained no citations. The Article 22 failure originated from retrieval, because the assistant cited retrieved excerpts but none from the expected article.

I would present the assistant to the data protection team as a prototype, but not deploy it for independent use. The team should understand that the assistant can help locate and summarise GDPR provisions, but its answers must be checked against the original legal text. 

## What did it cost?

Run the cell below to see the tokens and the cost of the lab.

In [12]:
log = pd.DataFrame(usage_log)
summary = log.groupby("step", sort=False)[["input", "output"]].sum()
summary["calls"] = log.groupby("step", sort=False).size()
summary["cost ($)"] = [cost(row.input, row.output) for row in summary.itertuples()]
print(f"Total: {len(log)} calls, ${summary['cost ($)'].sum():.4f}")
summary.round(5)

KeyError: 'step'

In [13]:
print(usage_log)
print(log.columns.tolist())

[]
[]


In [15]:
usage_log = []

In [17]:
client_openai = OpenAI()

def answer(question, k=5, where=RULES_ONLY):

    # 1. Retrieve relevant GDPR chunks
    retrieved_df = retrieve(question, k=k, where=where)

    # 2. Build the context
    context = "\n\n".join(
        f"[{row['id']}] {row['section']}, page {row['page']}\n{row['text']}"
        for _, row in retrieved_df.iterrows()
    )

    # 3. Call the OpenAI model
    response = client_openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"GDPR excerpts:\n\n{context}\n\nQuestion: {question}"
            }
        ],
        temperature=0
    )

    # 4. Track token usage
    usage_log.append({
        "step": "answers",
        "input": response.usage.prompt_tokens,
        "output": response.usage.completion_tokens
    })

    # 5. Extract the answer
    answer_text = response.choices[0].message.content

    # 6. Find citations
    cited_ids = list(dict.fromkeys(
        re.findall(r"\[(gdpr-\d+)\]", answer_text)
    ))

    # 7. Return the answer and metadata
    return {
        "answer": answer_text,
        "cited": cited_ids,
        "retrieved": retrieved_df["id"].tolist(),
        "sections": retrieved_df["section"].tolist()
    }

In [18]:
result = answer("Can I ask a company to delete my data?")

print(result["answer"])
print(usage_log)

Yes, you can ask a company to delete your data. Under Article 17, you have the right to obtain the erasure of personal data concerning you without undue delay, and the controller has the obligation to erase it under certain conditions [gdpr-261].
[{'step': 'answers', 'input': 1152, 'output': 53}]


In [19]:
for question in TEST_SET["question"]:
    answer(question)

for question in OUT_OF_SCOPE:
    answer(question)

print("Total logged calls:", len(usage_log))

Total logged calls: 13


In [20]:
log = pd.DataFrame(usage_log)

summary = log.groupby("step", sort=False)[["input", "output"]].sum()
summary["calls"] = log.groupby("step", sort=False).size()

summary["cost ($)"] = [
    cost(row.input, row.output)
    for row in summary.itertuples()
]

print(f"Total: {len(log)} calls, ${summary['cost ($)'].sum():.4f}")

display(summary.round(5))

Total: 13 calls, $0.0038


,input,output,calls,cost ($)
step,,,,
answers,14952,646,13,0.0038


**Checkpoint:** 15 calls and about half a US cent. Indexing with MiniLM was free; every answer costs one call of about 1,100 input tokens.

## Extra: tune the chunk size, or use your own document

Choose one:

+ **Chunk size.** Build one collection per chunk size (100, 150, 200 and 400 words, with an overlap of a fifth of the size) and compare their hit rate at k = 5 with `RULES_ONLY`. Which size works best on the GDPR, and is it the same as for the AI Act in the lesson? Be careful: with only 10 questions, a difference of one question is 0.1.
+ **Your own document.** Index a PDF you are allowed to use (your company's public privacy policy, a manual, a public report), write five questions with the page that answers them, and measure the hit rate. What did you have to change in `load_pdf` and `chunk_document`?

In [21]:
# Your code here
chunk_sizes = [100, 150, 200, 400]

# Use a new in-memory ChromaDB client
experiment_client = chromadb.Client()

results = []

for size in chunk_sizes:

    overlap = size // 5

    # 1. Create chunks
    test_chunks = chunk_document(
        reader.pages,
        size=size,
        overlap=overlap
    )

    # 2. Generate normalized embeddings
    texts = [chunk["text"] for chunk in test_chunks]

    embeddings = embedder.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # 3. Create a separate ChromaDB collection
    collection_name = f"gdpr_{size}"

    test_collection = experiment_client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    # 4. Add chunks in batches
    batch_size = 500

    for i in range(0, len(test_chunks), batch_size):

        batch = test_chunks[i:i + batch_size]

        test_collection.add(
            ids=[c["id"] for c in batch],
            documents=[c["text"] for c in batch],
            embeddings=embeddings[i:i + batch_size].tolist(),
            metadatas=[
                {"page": c["page"], "section": c["section"]}
                for c in batch
            ]
        )

    # 5. Evaluate hit rate at k=5 with RULES_ONLY
    hits = 0
    missed = []

    for _, row in TEST_SET.iterrows():

        question_embedding = embedder.encode(
            row["question"],
            normalize_embeddings=True
        ).tolist()

        search_results = test_collection.query(
            query_embeddings=[question_embedding],
            n_results=5,
            where=RULES_ONLY,
            include=["metadatas"]
        )

        retrieved_sections = [
            m["section"]
            for m in search_results["metadatas"][0]
        ]

        if row["expected"] in retrieved_sections:
            hits += 1
        else:
            missed.append(row["question"])

    hit_rate_value = hits / len(TEST_SET)

    results.append({
        "chunk_size": size,
        "overlap": overlap,
        "n_chunks": len(test_chunks),
        "hits": hits,
        "hit_rate": hit_rate_value,
        "missed_questions": missed
    })

# 6. Compare results
comparison_df = pd.DataFrame(results)

display(comparison_df)

,chunk_size,overlap,n_chunks,hits,hit_rate,missed_questions
0,100,20,698,8,0.8,"[What are the legal grounds for processing personal data?, Can I move my data from one..."
1,150,30,465,8,0.8,"[Can I move my data from one service provider to another?, Can a decision about me be ..."
2,200,40,349,8,0.8,"[Can I move my data from one service provider to another?, Can a decision about me be ..."
3,400,80,175,7,0.7,"[What are the legal grounds for processing personal data?, At what age can a child con..."


**Conclusions:** after comparing different chunk sizes, the 100, 200 and 150 chunk sizes work equally well, with a hit rate of 0.8. I would select 150 words as a reasonable compromise between retrieval performance, computational efficiency and the risk of token truncation.